# Validação do ambiente U-Mamba (estágio 11)

Esta etapa verifica se o ambiente de GPU é adequado para integrar a implementação oficial U-Mamba. O objetivo é realizar um smoke test de construção e inferência, sem iniciar ainda o treinamento completo.

## Requisitos oficiais

O repositório oficial `bowang-lab/U-Mamba` documenta Ubuntu 20.04, Python 3.10, CUDA 11.8, PyTorch 2.0.1, `causal-conv1d` e `mamba-ssm`. Esta checagem evita alterar silenciosamente um runtime incompatível.

In [ ]:
import platform
import sys

import torch

print(f"Sistema: {platform.platform()}")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA do PyTorch: {torch.version.cuda}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


## Diagnóstico de dependências

Verifica se `mamba_ssm`, `causal_conv1d` e a classe oficial `UMambaEnc_2d` já estão disponíveis.

In [ ]:
import importlib.util

modules = {
    "mamba_ssm": importlib.util.find_spec("mamba_ssm"),
    "causal_conv1d": importlib.util.find_spec("causal_conv1d"),
    "nnunetv2": importlib.util.find_spec("nnunetv2"),
}
for name, spec in modules.items():
    print(f"{name}: {'OK' if spec is not None else 'AUSENTE'}")


## Instalação oficial — executar somente em runtime dedicado

A célula abaixo fica desativada por padrão. Ela deve ser habilitada somente em um runtime separado do baseline e compatível com os requisitos da implementação oficial. A instalação pode exigir reinício do runtime.

In [ ]:
RUN_OFFICIAL_INSTALL = False

if RUN_OFFICIAL_INSTALL:
    import pathlib
    import subprocess
    import sys

    if sys.version_info[:2] != (3, 10):
        raise RuntimeError(
            f"O ambiente oficial foi documentado para Python 3.10; atual={sys.version_info.major}.{sys.version_info.minor}. "
            "Use um runtime/ambiente Python 3.10 antes de prosseguir."
        )
    if not torch.cuda.is_available():
        raise RuntimeError("GPU CUDA não disponível.")

    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "torch==2.0.1", "torchvision==0.15.2",
        "--index-url", "https://download.pytorch.org/whl/cu118",
    ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "causal-conv1d>=1.2.0"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "mamba-ssm", "--no-cache-dir"], check=True)

    target = pathlib.Path("/content/U-Mamba")
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/bowang-lab/U-Mamba", str(target)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(target / "umamba")], check=True)
    print("Instalação concluída. Reinicie o runtime antes do smoke test, se necessário.")
else:
    print("Instalação desativada. Altere RUN_OFFICIAL_INSTALL para True apenas em runtime dedicado.")


## Bootstrap do TCC

Carrega o adaptador do projeto após a instalação oficial estar disponível.

In [ ]:
import importlib
import pathlib
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()


## Smoke test U-Mamba RGB

Quando o ambiente estiver pronto, constrói a classe oficial `UMambaEnc_2d` com 3 canais e executa uma inferência sintética 256×256. Isso comprova a integração arquitetural antes do treinamento.

In [ ]:
from src.models.umamba import build_official_umamba_enc_2d, umamba_available

if not umamba_available():
    print("U-Mamba ainda não disponível neste runtime. Execute a instalação em ambiente compatível.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_official_umamba_enc_2d(
        input_channels=3,
        num_classes=1,
        input_size=(256, 256),
        features_per_stage=(32, 64, 128, 256),
    ).to(device)
    x = torch.randn(1, 3, 256, 256, device=device)
    with torch.inference_mode():
        y = model(x)
    print(f"Entrada: {tuple(x.shape)}")
    print(f"Saída: {tuple(y.shape)}")
    print(f"Parâmetros: {sum(p.numel() for p in model.parameters()):,}")
    if tuple(y.shape) != (1, 1, 256, 256):
        raise RuntimeError(f"Shape de saída inesperado: {tuple(y.shape)}")
    print("Smoke test U-Mamba concluído.")


## Marco desta etapa

A integração é considerada tecnicamente validada quando as dependências oficiais importam, a rede é construída e o forward 256×256 termina com saída 1×1×256×256. O treinamento completo fica para a etapa seguinte e não deve ser iniciado se esse teste falhar.